<a href="https://colab.research.google.com/github/Carol-zolet/Carol-zolet/blob/main/fast_stable_diffusion_ComfyUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ComfyUI notebook by https://github.com/TheLastBen/fast-stable-diffusion**

In [1]:
#@markdown # Connect Google Drive
from google.colab import drive
from IPython.display import clear_output
import ipywidgets as widgets
import os

def inf(msg, style, wdth): inf = widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth));display(inf)
Shared_Drive = "" #@param {type:"string"}
#@markdown - Leave empty if you're not using a shared drive

print("[0;33mConnecting...")
drive.mount('/content/gdrive')

if Shared_Drive!="" and os.path.exists("/content/gdrive/Shareddrives"):
  mainpth="Shareddrives/"+Shared_Drive
else:
  mainpth="MyDrive"

clear_output()
inf('\u2714 Done','success', '50px')

#@markdown ---

Button(button_style='success', description='✔ Done', disabled=True, layout=Layout(min_width='50px'), style=But…

In [2]:
from google.colab import drive
import os

# Montar Drive
if not os.path.exists('/content/gdrive'):
    drive.mount('/content/gdrive')
    print("✅ Google Drive conectado!")
else:
    print("✅ Google Drive já conectado!")


✅ Google Drive já conectado!


In [3]:
#@markdown # Install/Update ComfyUI repo
from IPython.utils import capture
from IPython.display import clear_output
from subprocess import getoutput
import ipywidgets as widgets
import sys
import fileinput
import os
import time
import base64
import requests
from urllib.request import urlopen, Request
from urllib.parse import urlparse, parse_qs, unquote
from tqdm import tqdm
import six


if not os.path.exists("/content/gdrive"):
  print('[1;31mGdrive not connected, using temporary colab storage ...')
  time.sleep(4)
  mainpth="MyDrive"
  !mkdir -p /content/gdrive/$mainpth
  Shared_Drive=""

if Shared_Drive!="" and not os.path.exists("/content/gdrive/Shareddrives"):
  print('[1;31mShared drive not detected, using default MyDrive')
  mainpth="MyDrive"

with capture.capture_output() as cap:
  def inf(msg, style, wdth): inf = widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth));display(inf)
  fgitclone = "git clone --depth 1"
  !git clone -q --depth 1 --branch main https://github.com/TheLastBen/diffusers
  %cd /content/gdrive/$mainpth/
  !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI
  !mkdir -p /content/gdrive/$mainpth/sd/cache/
  os.environ['TRANSFORMERS_CACHE']=f"/content/gdrive/{mainpth}/sd/cache"
  os.environ['TORCH_HOME'] = f"/content/gdrive/{mainpth}/sd/cache"

with capture.capture_output() as cap:
  %cd /content/gdrive/$mainpth/ComfyUI
  !git reset --hard
  time.sleep(1)
  !git pull
clear_output()
inf('\u2714 Done','success', '50px')

#@markdown ---

Button(button_style='success', description='✔ Done', disabled=True, layout=Layout(min_width='50px'), style=But…

In [4]:
!nvidia-smi
print("\n" + "="*70)
print("⚠️  IMPORTANTE: Certifique-se de estar usando GPU T4 ou superior!")
print("   Menu → Ambiente de execução → Alterar tipo → T4 GPU")
print("="*70)

Sat Jan 24 23:57:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
#@markdown # Requirements

print('[1;32mInstalling requirements...')

with capture.capture_output() as cap:
  %cd /content/
  !wget -q -i https://raw.githubusercontent.com/TheLastBen/fast-stable-diffusion/main/Dependencies/A1111.txt
  !dpkg -i *.deb
  !pip uninstall wandb -y
  !tar -C / --zstd -xf gcolabdeps.tar.zst
  !rm *.deb | rm *.zst | rm *.txt

  if not os.path.exists('gdrive/'+mainpth+'/sd/libtcmalloc/libtcmalloc_minimal.so.4'):
    %env CXXFLAGS=-std=c++14
    !wget -q https://github.com/gperftools/gperftools/releases/download/gperftools-2.5/gperftools-2.5.tar.gz && tar zxf gperftools-2.5.tar.gz && mv gperftools-2.5 gperftools
    !wget -q https://github.com/TheLastBen/fast-stable-diffusion/raw/main/AUTOMATIC1111_files/Patch
    %cd /content/gperftools
    !patch -p1 < /content/Patch
    !./configure --enable-minimal --enable-libunwind --enable-frame-pointers --enable-dynamic-sized-delete-support --enable-sized-delete --enable-emergency-malloc; make -j4
    !mkdir -p /content/gdrive/$mainpth/sd/libtcmalloc && cp .libs/libtcmalloc*.so* /content/gdrive/$mainpth/sd/libtcmalloc
    %env LD_PRELOAD=/content/gdrive/$mainpth/sd/libtcmalloc/libtcmalloc_minimal.so.4
    %cd /content
    !rm *.tar.gz Patch && rm -r /content/gperftools
  else:
    %env LD_PRELOAD=/content/gdrive/$mainpth/sd/libtcmalloc/libtcmalloc_minimal.so.4

  !pip install wandb==0.15.12 pydantic==2.10.5 numpy==1.26 scipy==1.15.3 -qq
  !pip install diffusers accelerate transformers av comfyui-frontend-package comfyui-workflow-templates alembic -U -qq
  !rm -r /usr/local/lib/python3.12/dist-packages/tensorflow*
  os.environ['PYTHONWARNINGS'] = 'ignore'
  !sed -i 's@text = _formatwarnmsg(msg)@text =\"\"@g' /usr/lib/python3.12/warnings.py
  !sed -i 's@raise AttributeError(f"module {module!r} has no attribute {name!r}")@@g' /usr/local/lib/python3.12/dist-packages/jax/_src/deprecations.py
  !sed -i 's@globalns, localns, set()@globalns, localns, recursive_guard=set()@g' /usr/local/lib/python3.12/dist-packages/pydantic/typing.py

clear_output()
inf('\u2714 Done','success', '50px')

#@markdown ---

Button(button_style='success', description='✔ Done', disabled=True, layout=Layout(min_width='50px'), style=But…

In [6]:
import os
import subprocess

print("🔧 INSTALANDO COMFYUI + DEPENDÊNCIAS")
print("="*70)

# Definir diretório
comfyui_dir = "/content/gdrive/MyDrive/ComfyUI_Perrengue"

# Clonar ComfyUI se não existir
if not os.path.exists(comfyui_dir):
    print("\n📥 Clonando ComfyUI...")
    !git clone https://github.com/comfyanonymous/ComfyUI {comfyui_dir}
    print("✅ ComfyUI clonado!")
else:
    print("\n✅ ComfyUI já existe!")

# Criar link simbólico
if os.path.exists('/content/ComfyUI'):
    !rm -rf /content/ComfyUI
!ln -sf {comfyui_dir} /content/ComfyUI

os.chdir('/content/ComfyUI')

# Instalar dependências
print("\n📦 Instalando dependências do ComfyUI...")
!pip install -q -r requirements.txt

# Instalar extras
print("\n📦 Instalando bibliotecas extras...")
!pip install -q xformers
!pip install -q pyngrok
!pip install -q insightface
!pip install -q onnxruntime-gpu
!pip install -q opencv-python

print("\n✅ TODAS AS DEPENDÊNCIAS INSTALADAS!")
print("="*70)


🔧 INSTALANDO COMFYUI + DEPENDÊNCIAS

✅ ComfyUI já existe!

📦 Instalando dependências do ComfyUI...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 5.6 MB/s eta 0:00:00
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/packaging/requirements.py", line 36, in __init__
    parsed = _parse_requirement(requirement_string)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/packaging/_parser.py", line 62, in parse_requirement
    return _parse_requirement(Tokenizer(source, rules=DEFAULT_RULES))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/packaging/_parser.py", line 80, in _parse_requirement
    url, specifier, marker = _parse_requirement_details(tokenizer)
                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_v

In [7]:
import os

print("🔌 INSTALANDO CUSTOM NODES")
print("="*70)

nodes_dir = "/content/ComfyUI/custom_nodes"
os.makedirs(nodes_dir, exist_ok=True)

# Lista de custom nodes necessários
custom_nodes = {
    "ComfyUI-Manager": "https://github.com/ltdrdata/ComfyUI-Manager",
    "comfyui-reactor-node": "https://github.com/Gourieff/comfyui-reactor-node",
    "ComfyUI_UltimateSDUpscale": "https://github.com/ssitu/ComfyUI_UltimateSDUpscale",
    "comfyui_controlnet_aux": "https://github.com/Fannovel16/comfyui_controlnet_aux",
    "ComfyUI-Florence2": "https://github.com/kijai/ComfyUI-Florence2",
    "ComfyUI-Portrait-Master": "https://github.com/florestefano1975/ComfyUI-Portrait-Master"
}

for node_name, repo_url in custom_nodes.items():
    node_path = f"{nodes_dir}/{node_name}"

    if not os.path.exists(node_path):
        print(f"\n📥 Instalando {node_name}...")
        !git clone {repo_url} {node_path}

        # Instalar requirements se existir
        req_file = f"{node_path}/requirements.txt"
        if os.path.exists(req_file):
            print(f"   Instalando dependências de {node_name}...")
            !pip install -q -r {req_file}

        print(f"   ✅ {node_name} instalado!")
    else:
        print(f"✅ {node_name} já instalado!")

print("\n✅ TODOS OS CUSTOM NODES INSTALADOS!")
print("="*70)



🔌 INSTALANDO CUSTOM NODES
✅ ComfyUI-Manager já instalado!

📥 Instalando comfyui-reactor-node...
Cloning into '/content/ComfyUI/custom_nodes/comfyui-reactor-node'...
fatal: could not read Username for 'https://github.com': No such device or address
   ✅ comfyui-reactor-node instalado!
✅ ComfyUI_UltimateSDUpscale já instalado!
✅ comfyui_controlnet_aux já instalado!
✅ ComfyUI-Florence2 já instalado!
✅ ComfyUI-Portrait-Master já instalado!

✅ TODOS OS CUSTOM NODES INSTALADOS!


In [8]:
import os

print("📥 BAIXANDO MODELOS FLUX")
print("="*70)
print("⏳ Isso vai demorar 10-20 minutos na primeira vez")
print("   Próximas vezes será instantâneo (salvo no Drive)")
print("="*70)

base_dir = "/content/gdrive/MyDrive/ComfyUI_Perrengue/models"

# Criar diretórios
folders = ['unet', 'clip', 'vae', 'loras', 'upscale_models', 'insightface', 'controlnet']
for folder in folders:
    os.makedirs(f"{base_dir}/{folder}", exist_ok=True)

# 1. Flux Dev fp8 (~5.5 GB)
print("\n[1/8] Flux Dev fp8...")
flux_path = f"{base_dir}/unet/flux1-dev-fp8.safetensors"
if not os.path.exists(flux_path):
    !wget -c --show-progress -O {flux_path} \
        "https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-dev-fp8.safetensors"
    print("✅ Flux baixado!")
else:
    print("✅ Flux já existe!")

# 2. T5-XXL (~9.5 GB)
print("\n[2/8] T5-XXL CLIP...")
t5_path = f"{base_dir}/clip/t5xxl_fp16.safetensors"
if not os.path.exists(t5_path):
    !wget -c --show-progress -O {t5_path} \
        "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors"
    print("✅ T5 baixado!")
else:
    print("✅ T5 já existe!")

# 3. CLIP-L (~246 MB)
print("\n[3/8] CLIP-L...")
clip_path = f"{base_dir}/clip/clip_l.safetensors"
if not os.path.exists(clip_path):
    !wget -c --show-progress -O {clip_path} \
        "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors"
    print("✅ CLIP-L baixado!")
else:
    print("✅ CLIP-L já existe!")

# 4. VAE (~335 MB)
print("\n[4/8] VAE...")
vae_path = f"{base_dir}/vae/ae.safetensors"
if not os.path.exists(vae_path):
    !wget -c --show-progress -O {vae_path} \
        "https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors"
    print("✅ VAE baixado!")
else:
    print("✅ VAE já existe!")

# 5. InSwapper (Face Swap) (~554 MB)
print("\n[5/8] InSwapper (ReActor)...")
inswapper_path = f"{base_dir}/insightface/inswapper_128.onnx"
if not os.path.exists(inswapper_path):
    !wget -c --show-progress -O {inswapper_path} \
        "https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx"
    print("✅ InSwapper baixado!")
else:
    print("✅ InSwapper já existe!")

# 6. Upscaler 4x (~67 MB)
print("\n[6/8] 4x-UltraSharp Upscaler...")
upscale_path = f"{base_dir}/upscale_models/4x-UltraSharp.pth"
if not os.path.exists(upscale_path):
    !wget -c --show-progress -O {upscale_path} \
        "https://huggingface.co/Kim2091/UltraSharp/resolve/main/4x-UltraSharp.pth"
    print("✅ Upscaler baixado!")
else:
    print("✅ Upscaler já existe!")

# 7. RealESRGAN (~64 MB)
print("\n[7/8] RealESRGAN...")
real_path = f"{base_dir}/upscale_models/RealESRGAN_x4plus.pth"
if not os.path.exists(real_path):
    !wget -c --show-progress -O {real_path} \
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"
    print("✅ RealESRGAN baixado!")
else:
    print("✅ RealESRGAN já existe!")

# 8. GFPGAN para face restore
print("\n[8/8] GFPGAN Face Restoration...")
gfpgan_path = f"{base_dir}/upscale_models/GFPGANv1.4.pth"
if not os.path.exists(gfpgan_path):
    !wget -c --show-progress -O {gfpgan_path} \
        "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth"
    print("✅ GFPGAN baixado!")
else:
    print("✅ GFPGAN já existe!")

print("\n" + "="*70)
print("✅ TODOS OS MODELOS BASE BAIXADOS!")
print(f"📦 Total: ~16 GB")
print(f"💾 Salvos em: {base_dir}")
print("="*70)

📥 BAIXANDO MODELOS FLUX
⏳ Isso vai demorar 10-20 minutos na primeira vez
   Próximas vezes será instantâneo (salvo no Drive)

[1/8] Flux Dev fp8...
✅ Flux já existe!

[2/8] T5-XXL CLIP...
✅ T5 já existe!

[3/8] CLIP-L...
✅ CLIP-L já existe!

[4/8] VAE...
✅ VAE já existe!

[5/8] InSwapper (ReActor)...
✅ InSwapper já existe!

[6/8] 4x-UltraSharp Upscaler...
✅ Upscaler já existe!

[7/8] RealESRGAN...
✅ RealESRGAN já existe!

[8/8] GFPGAN Face Restoration...
✅ GFPGAN já existe!

✅ TODOS OS MODELOS BASE BAIXADOS!
📦 Total: ~16 GB
💾 Salvos em: /content/gdrive/MyDrive/ComfyUI_Perrengue/models


In [11]:
#@markdown # Model Download/Load

import gdown
from gdown.download import get_url_from_gdrive_confirmation
import re

Use_Temp_Storage = False #@param {type:"boolean"}
#@markdown - If not, make sure you have enough space on your gdrive

#@markdown ---

Model_Version = "SDXL" #@param ["SDXL", "1.5", "v1.5 Inpainting", "flux"]

#@markdown Or
MODEL_LINK = "" #@param {type:"string"}


def getsrc(url):
    parsed_url = urlparse(url)
    if parsed_url.netloc == 'civitai.com':
        src='civitai'
    elif parsed_url.netloc == 'drive.google.com':
        src='gdrive'
    elif parsed_url.netloc == 'huggingface.co':
        src='huggingface'
    else:
        src='others'
    return src

src=getsrc(MODEL_LINK)

def get_name(url, gdrive):
    if not gdrive:
        response = requests.get(url, allow_redirects=False)
        if "Location" in response.headers:
            redirected_url = response.headers["Location"]
            quer = parse_qs(urlparse(redirected_url).query)
            if "response-content-disposition" in quer:
                disp_val = quer["response-content-disposition"][0].split(";")
                for vals in disp_val:
                    if vals.strip().startswith("filename="):
                        filenm=unquote(vals.split("=", 1)[1].strip())
                        return filenm.replace("\"","")
    else:
        headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36"}
        lnk="https://drive.google.com/uc?id={id}&export=download".format(id=url[url.find("/d/")+3:url.find("/view")])
        res = requests.session().get(lnk, headers=headers, stream=True, verify=True)
        res = requests.session().get(get_url_from_gdrive_confirmation(res.text), headers=headers, stream=True, verify=True)
        content_disposition = six.moves.urllib_parse.unquote(res.headers["Content-Disposition"])
        filenm = re.search('attachment; filename="(.*?)"', content_disposition).groups()[0]
        return filenm


def dwn(url, dst, msg):
    file_size = None
    req = Request(url, headers={"User-Agent": "torch.hub"})
    u = urlopen(req)
    meta = u.info()
    if hasattr(meta, 'getheaders'):
        content_length = meta.getheaders("Content-Length")
    else:
        content_length = meta.get_all("Content-Length")
    if content_length is not None and len(content_length) > 0:
        file_size = int(content_length[0])

    with tqdm(total=file_size, disable=False, mininterval=0.5,
              bar_format=msg+' |{bar:20}| {percentage:3.0f}%') as pbar:
        with open(dst, "wb") as f:
            while True:
                buffer = u.read(8192)
                if len(buffer) == 0:
                    break
                f.write(buffer)
                pbar.update(len(buffer))
            f.close()


def sdmdls(ver, Use_Temp_Storage):

  if ver=='1.5':
    if Use_Temp_Storage:
      os.makedirs('/content/temp_models', exist_ok=True)
      model='/content/temp_models/v1-5-pruned-emaonly.safetensors'
    else:
      model='/content/gdrive/'+mainpth+'/ComfyUI/models/checkpoints/v1-5-pruned-emaonly.safetensors'
    link='https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors'
  elif ver=='flux':
    if Use_Temp_Storage:
      os.makedirs('/content/temp_models', exist_ok=True)
      model='/content/temp_models/flux1-dev-fp8.safetensors'
    else:
      model='/content/gdrive/'+mainpth+'/ComfyUI/models/checkpoints/flux1-dev-fp8.safetensors'
    link='https://huggingface.co/lllyasviel/flux1_dev/resolve/main/flux1-dev-fp8.safetensors'
  elif ver=='v1.5 Inpainting':
    if Use_Temp_Storage:
      os.makedirs('/content/temp_models', exist_ok=True)
      model='/content/temp_models/sd-v1-5-inpainting.ckpt'
    else:
      model='/content/gdrive/'+mainpth+'/ComfyUI/models/checkpoints/sd-v1-5-inpainting.ckpt'
    link='https://huggingface.co/runwayml/stable-diffusion-inpainting/resolve/main/sd-v1-5-inpainting.ckpt'
  elif ver=='SDXL':
    if Use_Temp_Storage:
      os.makedirs('/content/temp_models', exist_ok=True)
      model='/content/temp_models/sd_xl_base_1.0.safetensors'
    else:
      model='/content/gdrive/'+mainpth+'/ComfyUI/models/checkpoints/sd_xl_base_1.0.safetensors'
    link='https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors'

  if not os.path.exists(model):
    !gdown --fuzzy -O $model $link
    if os.path.exists(model):
      clear_output()
      inf('\u2714 Done','success', '50px')
    else:
      inf('\u2718 Something went wrong, try again','danger', "250px")
  else:
      clear_output()
      inf('\u2714 Model already exists','primary', '300px')

  return model


if MODEL_LINK != "":

      if src=='civitai':
         modelname=get_name(MODEL_LINK, False)
         if Use_Temp_Storage:
            os.makedirs('/content/temp_models', exist_ok=True)
            model=f'/content/temp_models/{modelname}'
         else:
            model=f'/content/gdrive/{mainpth}/ComfyUI/models/checkpoints/{modelname}'
         if not os.path.exists(model):
            dwn(MODEL_LINK, model, 'Downloading the custom model')
            clear_output()
         else:
            inf('\u2714 Model already exists','primary', '300px')
      elif src=='gdrive':
         modelname=get_name(MODEL_LINK, True)
         if Use_Temp_Storage:
            os.makedirs('/content/temp_models', exist_ok=True)
            model=f'/content/temp_models/{modelname}'
         else:
            model=f'/content/gdrive/{mainpth}/ComfyUI/models/checkpoints/{modelname}'
         if not os.path.exists(model):
            gdown.download(url=MODEL_LINK, output=model, quiet=False, fuzzy=True)
            clear_output()
         else:
            inf('\u2714 Model already exists','primary', '300px')
      else:
         modelname=os.path.basename(MODEL_LINK)
         if Use_Temp_Storage:
            os.makedirs('/content/temp_models', exist_ok=True)
            model=f'/content/temp_models/{modelname}'
         else:
            model=f'/content/gdrive/{mainpth}/ComfyUI/models/checkpoints/{modelname}'
         if not os.path.exists(model):
            gdown.download(url=MODEL_LINK, output=model, quiet=False, fuzzy=True)
            clear_output()
         else:
            inf('\u2714 Model already exists','primary', '700px')

      if os.path.exists(model) and os.path.getsize(model) > 1810671599:
        inf('\u2714 Model downloaded, using the custom model.','success', '300px')
      else:
        !rm model
        inf('\u2718 Wrong link, check that the link is valid','danger', "300px")

else:
  model=sdmdls(Model_Version, Use_Temp_Storage)

#@markdown ---

Button(button_style='primary', description='✔ Model already exists', disabled=True, layout=Layout(min_width='3…

In [15]:
from google.colab import files
import os

print("╔" + "═"*68 + "╗")
print("║  📤 UPLOAD DO LORA DA CAROLINE                                   ║")
print("╚" + "═"*68 + "╝")
print()
print("📁 Selecione o arquivo:")
print("   Palavra chave_ _caroline_1_.safetensors (292 MB)")
print()
print("⏳ Upload vai demorar ~2-5 minutos (dependendo da conexão)")
print()

lora_dir = "/content/gdrive/MyDrive/ComfyUI_Perrengue/models/loras"
os.makedirs(lora_dir, exist_ok=True)

# Upload
uploaded = files.upload()

# Salvar
for filename in uploaded.keys():
    print(f"\n⏳ Salvando: {filename}")

    dest_path = f"{lora_dir}/{filename}"
    with open(dest_path, 'wb') as f:
        f.write(uploaded[filename])

    size_mb = len(uploaded[filename]) / (1024*1024)

    print()
    print("╔" + "═"*68 + "╗")
    print("║  ✅ LORA DA CAROLINE INSTALADO COM SUCESSO!                     ║")
    print("╚" + "═"*68 + "╝")
    print()
    print(f"📁 Salvo em: {dest_path}")
    print(f"💾 Tamanho: {size_mb:.2f} MB")
    print(f"🎨 Nome no ComfyUI: {filename}")
    print()


╔════════════════════════════════════════════════════════════════════╗
║  📤 UPLOAD DO LORA DA CAROLINE                                   ║
╚════════════════════════════════════════════════════════════════════╝

📁 Selecione o arquivo:
   Palavra chave_ _caroline_1_.safetensors (292 MB)

⏳ Upload vai demorar ~2-5 minutos (dependendo da conexão)



In [13]:
# Verificar se o LoRA foi colocado no Drive
import os

lora_path = "/content/gdrive/MyDrive/ComfyUI_Perrengue/models/loras/Palavra chave_ _caroline_1_.safetensors"

if os.path.exists(lora_path):
    size = os.path.getsize(lora_path) / (1024**2)
    print("✅ LoRA da Caroline encontrado!")
    print(f"📁 Local: {lora_path}")
    print(f"💾 Tamanho: {size:.2f} MB")
else:
    print("❌ LoRA não encontrado!")
    print(f"📁 Esperado em: {lora_path}")
    print("\nColoque o arquivo manualmente no Google Drive:")
    print("1. Abra: drive.google.com")
    print("2. Vá para: Meu Drive/ComfyUI_Perrengue/models/loras/")
    print("3. Arraste o arquivo para lá")

✅ LoRA da Caroline encontrado!
📁 Local: /content/gdrive/MyDrive/ComfyUI_Perrengue/models/loras/Palavra chave_ _caroline_1_.safetensors
💾 Tamanho: 292.23 MB


In [12]:
#@markdown # Download LoRA

LoRA_LINK = "" #@param {type:"string"}

if LoRA_LINK == "":
  inf('\u2714 Nothing to do','primary', '200px')
else:
  os.makedirs('/content/gdrive/'+mainpth+'/ComfyUI/models/loras', exist_ok=True)

  src=getsrc(LoRA_LINK)

  if src=='civitai':
      modelname=get_name(LoRA_LINK, False)
      loramodel=f'/content/gdrive/{mainpth}/ComfyUI/models/loras/{modelname}'
      if not os.path.exists(loramodel):
        dwn(LoRA_LINK, loramodel, 'Downloading the LoRA model '+modelname)
        clear_output()
      else:
        inf('\u2714 Model already exists','primary', '200px')
  elif src=='gdrive':
      modelname=get_name(LoRA_LINK, True)
      loramodel=f'/content/gdrive/{mainpth}/ComfyUI/models/loras/{modelname}'
      if not os.path.exists(loramodel):
        gdown.download(url=LoRA_LINK, output=loramodel, quiet=False, fuzzy=True)
        clear_output()
      else:
        inf('\u2714 Model already exists','primary', '200px')
  else:
      modelname=os.path.basename(LoRA_LINK)
      loramodel=f'/content/gdrive/{mainpth}/ComfyUI/models/loras/{modelname}'
      if not os.path.exists(loramodel):
        gdown.download(url=LoRA_LINK, output=loramodel, quiet=False, fuzzy=True)
        clear_output()
      else:
        inf('\u2714 Model already exists','primary', '200px')

  if os.path.exists(loramodel) :
    inf('\u2714 LoRA downloaded','success', '200px')
  else:
    inf('\u2718 Wrong link, check that the link is valid','danger', "300px")

#@markdown ---

Button(button_style='primary', description='✔ Nothing to do', disabled=True, layout=Layout(min_width='200px'),…

In [ ]:
import os

print("📥 BAIXANDO LORAS DE MELHORAMENTO (OPCIONAL)")
print("="*70)
print("⏳ Isso vai demorar ~5-10 minutos")
print("="*70)

lora_dir = "/content/gdrive/MyDrive/ComfyUI_Perrengue/models/loras"

# Lista de LoRAs úteis para NSFW
loras = {
    "perfect_hands": {
        "url": "https://civitai.com/api/download/models/123456",  # Ajustar URL
        "filename": "perfect_hands.safetensors"
    },
    # Adicione mais LoRAs conforme necessário
}

print("\n⚠️  Configure os LoRAs manualmente ou pule esta célula")
print("   Você pode adicionar LoRAs depois via ComfyUI Manager")

In [ ]:
import subprocess
import threading
import time
import requests
import os

# ═══════════════════════════════════════
# ⚠️  INSIRA SEU TOKEN NGROK AQUI:
# ═══════════════════════════════════════
NGROK_TOKEN = "31vtbTbvKE01X9yoZ34nnzKpQqq_2gMHTRsBjUaU7N3HjkZBc"
# ═══════════════════════════════════════

if NGROK_TOKEN == "COLE_SEU_TOKEN_AQUI":
    print("❌ ERRO: Token Ngrok não configurado!")
    print()
    print("Como conseguir o token (GRÁTIS):")
    print("1. Acesse: https://ngrok.com")
    print("2. Faça cadastro (Sign up)")
    print("3. Copie seu token em:")
    print("   https://dashboard.ngrok.com/get-started/your-authtoken")
    print("4. Cole acima onde está o token")
    print("5. Execute esta célula novamente")
else:
    print("⇇ INICIANDO COMFYUI")
    print("="*70)

    # Matar processos anteriores
    print("\n1″️ Limpando processos anteriores...")
    !pkill -f "main.py" 2>/dev/null
    !pkill -f "ngrok" 2>/dev/null
    time.sleep(3)
    print("   ✔ Processos limpos")

    # Definir diretório do ComfyUI
    comfyui_path = "/content/gdrive/MyDrive/ComfyUI_Perrengue"
    print(f"\n2″️ Verificando estrutura em: {comfyui_path}")
    if not os.path.exists(comfyui_path):
        print(f"   ❌ ERRO: Diretório do ComfyUI não encontrado em {comfyui_path}!")
        print("   Certifique-se de ter executado as células anteriores para instalar o ComfyUI.")
    else:
        print("   ✔ Diretório encontrado.")

        # Iniciar ComfyUI
        print("\n3″️ Iniciando ComfyUI...")
        print("   (Isso pode demorar 30-90 segundos)")

        # Iniciar processo
        process = subprocess.Popen(
            ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            cwd=comfyui_path # Usar o caminho direto
        )

        # Aguardar estar pronto
        print("\n4″️ Aguardando ComfyUI inicializar...")

        max_wait = 120
        waited = 0
        ready = False

        while waited < max_wait:
            try:
                response = requests.get('http://localhost:8188', timeout=2)
                if response.status_code == 200:
                    ready = True
                    print(f"\n   ✔ ComfyUI pronto após {waited} segundos!")
                    break
            except:
                pass

            if waited % 10 == 0:
                print(f"   ⏳ {waited}s / {max_wait}s...")

            if process.poll() is not None:
                print("\n   ❌ Processo terminou inesperadamente!")
                break

            time.sleep(1)
            waited += 1

        if not ready:
            print("\n   ❌ ComfyUI não iniciou!")
            print("\n   Tente reiniciar o runtime e executar novamente")
        else:
            # Configurar Ngrok
            print("\n5″️ Configurando tùnel Ngrok...")

            !pip install -q pyngrok
            from pyngrok import ngrok

            ngrok.kill()
            ngrok.set_auth_token(NGROK_TOKEN)
            public_url = ngrok.connect(8188)

            print("\n" + "="*70)
            print("✔ COMFYUI RODANDO COM SUCESSO!")
            print("="*70)
            print()
            print(f"🌐 ACESSE AGORA:")
            print(f"   {public_url}")
            print()
            print("📋 PRÓXIMOS PASSOS:")
            print("   1. Clique no link acima")
            print("   2. Se aparecer aviso 'Visit Site', clique")
            print("   3. Importe o workflow 'Saindo do Perrengue'")
            print("   4. Configure o LoRA da Caroline")
            print("   5. Ajuste o prompt")
            print("   6. Queue Prompt!")
            print()
            print("⚠️  IMPORTANTE:")
            print("   • NÃO FECHE esta célula!")
            print("   • Sessão expira após ~12h")
            print("   • Salve suas imagens no Drive!")
            print()
            print("="*70)

            # Manter rodando
            print("\n⏳ Mantendo ComfyUI ativo...")
            print("   (Para parar: Runtime → Interrupt execution)")

            try:
                while True:
                    time.sleep(10)
                    if process.poll() is not None:
                        print("\n❌ ComfyUI parou de responder!")
                        break
            except KeyboardInterrupt:
                print("\n⏹ Parando ComfyUI...")
                process.terminate()
                ngrok.kill()
                print("✔ ComfyUI parado")

In [14]:
#@markdown # Start ComfyUI
from IPython.utils import capture
import time
import sys
import fileinput
import re
from subprocess import call


Ngrok_Token = "31vtbTbvKE01X9yoZ34nnzKpQqq_2gMHTRsBjUaU7N3HjkZBc" #@param {type:"string"}

#@markdown - Ngrok token must be entered to be able to run ComfyUI

#ngrok.kill()
#time.sleep(2)
!pip install -q pyngrok
from pyngrok import ngrok, conf
with capture.capture_output() as cap:
  localurl=ngrok.connect(666, pyngrok_config=conf.PyngrokConfig(auth_token=Ngrok_Token) , bind_tls=True).public_url
  # Corrected path for custom_nodes
  !rm -r /content/gdrive/MyDrive/ComfyUI_Perrengue/custom_nodes/.ipynb_checkpoints
# Corrected path for server.py
call("sed -i 's@^            if verbose:@@' /content/gdrive/MyDrive/ComfyUI_Perrengue/server.py", shell=True)
call("sed -i 's@^                logging.info(\"To see the GUI go to: {}://{}:{}\".format(scheme, address_print, port))@        logging.info(\"\u001b[32m\u2714 Connected\");logging.info(\"\u001b[1;34m"+localurl+"\u001b[0m\")@' /content/gdrive/MyDrive/ComfyUI_Perrengue/server.py", shell=True)
!sed -i 's@https:.*@{localurl}\u001b[0m\")@' /content/gdrive/MyDrive/ComfyUI_Perrengue/server.py
# Corrected path for main.py
!python /content/gdrive/MyDrive/ComfyUI_Perrengue/main.py --listen --port 666 #--preview-method auto

    torch (>=1.9.*)
           ~~~~~~^
    numpy (>=1.19.*) ; python_version >= "3.7"
           ~~~~~~~^
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-01-25 00:55:52.486
** Platform: Linux
** Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/gdrive/MyDrive/ComfyUI_Perrengue
** ComfyUI Base Folder Path: /content/gdrive/MyDrive/ComfyUI_Perrengue
** User directory: /content/gdrive/MyDrive/ComfyUI_Perrengue/user
** ComfyUI-Manager config path: /content/gdrive/MyDrive/ComfyUI_Perrengue/user/__manager/config.ini
** Log path: /content/gdrive/MyDrive/ComfyUI_Perrengue/user/comfyui.log

Prestartup times for custom nodes:
   8.0 seconds: /content/gdrive/MyDrive/ComfyUI_Perrengue/custom_nodes/ComfyUI-Manager

Checkpoint files will always be loaded safely.
Total VRAM 15095 MB, total RAM 12976 MB
pytorch version: 2.9.0+cu126
xformers version

In [ ]:
import os

print("📦 VERIFICANDO ARQUIVOS INSTALADOS")
print("="*70)

base_dir = "/content/gdrive/MyDrive/ComfyUI_Perrengue/models"

# Estrutura de verificação
checks = {
    "Modelos Base": {
        "Flux Dev fp8": f"{base_dir}/unet/flux1-dev-fp8.safetensors",
        "T5-XXL": f"{base_dir}/clip/t5xxl_fp16.safetensors",
        "CLIP-L": f"{base_dir}/clip/clip_l.safetensors",
        "VAE": f"{base_dir}/vae/ae.safetensors"
    },
    "Face Swap": {
        "InSwapper": f"{base_dir}/insightface/inswapper_128.onnx"
    },
    "Upscalers": {
        "4x-UltraSharp": f"{base_dir}/upscale_models/4x-UltraSharp.pth",
        "RealESRGAN": f"{base_dir}/upscale_models/RealESRGAN_x4plus.pth",
        "GFPGAN": f"{base_dir}/upscale_models/GFPGANv1.4.pth"
    },
    "LoRAs": {
        "Caroline": f"{base_dir}/loras/Palavra chave_ _caroline_1_.safetensors"
    }
}

all_ok = True

for category, files in checks.items():
    print(f"\n📁 {category}:")
    for name, path in files.items():
        if os.path.exists(path):
            size = os.path.getsize(path) / (1024**2)
            print(f"   ✅ {name}: {size:.1f} MB")
        else:
            print(f"   ❌ {name}: NÃO ENCONTRADO")
            all_ok = False

print("\n" + "="*70)
if all_ok:
    print("✅ TODOS OS ARQUIVOS OK!")
else:
    print("⚠️  Alguns arquivos faltam. Execute as células de download.")
print("="*70)

In [ ]:
from google.colab import files
import os

output_dir = "/content/gdrive/MyDrive/ComfyUI_Perrengue/output"

if os.path.exists(output_dir):
    print("📥 Baixando imagens geradas...")
    for filename in os.listdir(output_dir):
        if filename.endswith(('.png', '.jpg', '.jpeg')):
            filepath = os.path.join(output_dir, filename)
            print(f"   Baixando: {filename}")
            files.download(filepath)
    print("✅ Download concluído!")
else:
    print("❌ Nenhuma imagem gerada ainda!")
    print("   Gere imagens primeiro no ComfyUI")